In [ ]:
!pip install kfp 
!pip install kfp[kubernetes]
!pip install tensorboardX
!pip list | grep kfp

In [ ]:
import kfp
from kfp import dsl
from kfp.components import load_component_from_text
import os
from kfp import kubernetes

## Normal one

In [ ]:


anomaly_detection_op = load_component_from_text("""
name: Anomaly Detection Component
description: A component to run anomaly detection.
inputs:
  - {name: prometheus_url, type: String}
  - {name: polling_interval, type: Integer}
  - {name: name, type: String}
  - {name: type, type: String}  
outputs: []
implementation:
  container:
    image: fatemehbozorgi/test-repo:latest
    command: ["python", "main.py"]
    args: [
      "--prometheus-url", {inputValue: prometheus_url},
      "--polling-interval", {inputValue: polling_interval},
      "--type", {inputValue: type},
      "--name", {inputValue: name}
      
    ]
    
""")


@dsl.pipeline(
    name="Anomaly Detection Pipeline",
    description="A pipeline to detect anomalies in data."
)
def anomaly_detection_pipeline(prometheus_url: str, polling_interval: int, name: str, type_: str):  # Changed 'Type' to 'type_'
    anomaly_detection_task = anomaly_detection_op(
        prometheus_url=prometheus_url,
        polling_interval=polling_interval,
        name=name,
        type=type_  # Match the YAML input "type"
    )

if __name__ == "__main__":
    client = kfp.Client()
    client.create_run_from_pipeline_func(
        anomaly_detection_pipeline,
        arguments={
            "prometheus_url": "http://141.5.107.135:30090/",
            "polling_interval": 10,
            "name": "cp",
            "type_": "node"  # Match the new variable name in the function signature
        }
    )



## testing

In [ ]:
import kfp
from kfp import dsl
from kfp.components import load_component_from_text
import os

anomaly_detection_op = load_component_from_text("""
name: Anomaly Detection Component
description: A component to run anomaly detection.
inputs:
  - {name: prometheus_url, type: String}
  - {name: polling_interval, type: Integer}
  - {name: name, type: String}
  - {name: type, type: String}  
outputs:
  - {name: data, type: Artifact}
implementation:
  container:
    image: fatemehbozorgi/test-repo:latest
    command: ["python", "main.py"]
    args: [
      "--prometheus-url", {inputValue: prometheus_url},
      "--polling-interval", {inputValue: polling_interval},
      "--type", {inputValue: type},
      "--name", {inputValue: name},
      "--log-dir", {outputPath: data} 
    ]

""")

@dsl.component
def artifact_consumer(model: Input[Artifact]):
    print(model)
@dsl.pipeline(
    name="Anomaly Detection Pipeline",
    description="A pipeline to detect anomalies in data."
)
def anomaly_detection_pipeline(prometheus_url: str, polling_interval: int, name: str, type_: str):  
    # pvc1 = kubernetes.CreatePVC(
    #     pvc_name_suffix='-my-pvc1',
    #     access_modes=['ReadWriteMany'],
    #     size='5Gi',
    #     storage_class_name='microk8s-hostpath',
    # )
    anomaly_task  = anomaly_detection_op(
        prometheus_url=prometheus_url,
        polling_interval=polling_interval,
        name=name,
        type=type_     )
    # kubernetes.mount_pvc(
    #     anomaly_task,
    #     pvc_name=pvc1.outputs['name'],
    #     mount_path='/mnt/data',
    # )
artifact_consumer(model=anomaly_detection_op.output)
if __name__ == "__main__":
    client = kfp.Client()
    client.create_run_from_pipeline_func(
        anomaly_detection_pipeline,
        arguments={
            "prometheus_url": "http://141.5.107.135:30090/",
            "polling_interval": 10,
            "name": "cp",
            "type_": "node" 
        }
    )

In [ ]:
from kfp import dsl
from minio import Minio
from minio.error import S3Error
import os
import json
import time
from datetime import datetime

# Initialize the MinIO client
minio_client = Minio(
    "141.5.107.135:31001",  # Endpoint
    access_key="minio",
    secret_key="minio123",
    secure=False  # Set to True if using HTTPS
)

bucket_name = "logs-buket"  

@dsl.component(packages_to_install=["tensorflow", "minio"])
def upload_task():
    import os
    import time
    import tensorflow as tf
    from datetime import datetime
    from minio import Minio
    from minio.error import S3Error

    # Initialize the MinIO client
    minio_client = Minio(
        "10.1.242.178:9000",  # Endpoint
        access_key="minio",
        secret_key="minio123",
        secure=False  # Set to True if using HTTPS
    )

    bucket_name = "logs-buket"
    log_dir = "./tensorboard_logs"
    interval_seconds = 30
    steps = 1  # Initialize step counter

    # Create local directory for logs if it doesn't exist
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)
        print(f"Created log directory: {log_dir}")

    # TensorBoard writer
    writer = tf.summary.create_file_writer(log_dir)

    try:
        print(f"Starting log generation every {interval_seconds} seconds.")
        while steps <= 3:  # Generate 3 log events as an example
            # Generate some metrics (e.g., loss and accuracy) for this step
            loss = 0.1 * steps  # Dummy loss value
            accuracy = 0.9 + (steps * 0.01)  # Dummy accuracy value
            print(loss, accuracy)
            # Write metrics to TensorBoard summary
            with writer.as_default():
                
                tf.summary.scalar("loss", loss, step=steps)
                tf.summary.scalar("accuracy", accuracy, step=steps)

            print(f"Generated TensorBoard logs for step {steps}.")

            time.sleep(interval_seconds)
            steps += 1
    except KeyboardInterrupt:
        print("\nLog generation stopped.")

    print("Finished log generation.")
    

    # Upload the TensorBoard logs to MinIO
    for root, dirs, files in os.walk(log_dir):
        for file_name in files:
            local_path = os.path.join(root, file_name)
            object_name = os.path.relpath(local_path, log_dir)  # Relative path in MinIO
            try:
                minio_client.fput_object(
                    bucket_name=bucket_name,
                    object_name=object_name,
                    file_path=local_path,
                )
                print(f"Uploaded '{local_path}' to bucket '{bucket_name}' as '{object_name}'.")
            except S3Error as e:
                print(f"Failed to upload '{local_path}': {e}")

@dsl.pipeline(name="Upload Logs to MinIO", description="A pipeline to upload logs to MinIO.")
def pipeline_upload_logs():
    upload_task()

if __name__ == "__main__":
    log_directory = "./data"  # Local directory to store logs
    directory_path="./data" 
    
    # Generate logs and upload to MinIO on script termination
    # generate_event_files(log_directory,30)

    # To run as a Kubeflow pipeline
    from kfp import Client
    client = kfp.Client()
    client.create_run_from_pipeline_func(pipeline_upload_logs)


In [ ]:
from tensorboardX import SummaryWriter
import numpy as np
from minio import Minio
import os
import time

# Create a SummaryWriter to log data to a local directory
local_log_dir = 'simple_plot'
writer = SummaryWriter(local_log_dir)

# Generate x and y data
x = np.linspace(0, 10, 100)
y = np.sin(x)

# Log each data point as a scalar
for xi, yi in zip(x, y):
    writer.add_scalars('y_values', {'sin': yi}, xi)

# Close the writer after logging is done
writer.close()

# Initialize MinIO client
minio_client = Minio(
    "10.1.242.178:9000",  # MinIO endpoint
    access_key="minio",
    secret_key="minio123",
    secure=False  # Set to True if using HTTPS
)

# Check if the bucket exists, if not create it
bucket_name = "tensorboard-logs"
if not minio_client.bucket_exists(bucket_name):
    minio_client.make_bucket(bucket_name)

# Upload event files to MinIO
for root, dirs, files in os.walk(local_log_dir):
    for file in files:
        local_file = os.path.join(root, file)
        minio_file = f"tensorboard/{file}"
        print(f"Uploading {local_file} to MinIO as {minio_file}")
        minio_client.fput_object(bucket_name, minio_file, local_file)

# Optionally, clean up the local directory after upload
# shutil.rmtree(local_log_dir)


## simple connection to tensorboard

In [ ]:
from tensorboardX import SummaryWriter
import numpy as np

# Create a SummaryWriter to log data
writer = SummaryWriter('simple_plot')

# Generate x and y data
x = np.linspace(0, 10, 100)
y = np.sin(x)

# Log each data point as a scalar
for xi, yi in zip(x, y):
    writer.add_scalars('y_values', {'sin': yi}, xi)

# Close the writer after logging is done
writer.close()

